In [1]:
import ipywidgets as widgets
import pandas as pd
from IPython.display import display


class StringGrid(widgets.VBox):
    def __init__(
        self,
        df: pd.DataFrame,
        cell_width: str = "140px",
        index_width: str = "120px",
        height: str = "300px",
        width: str = "800px",
        show_index: bool = True,
    ):
        super().__init__()
        self.cell_width = cell_width
        self.index_width = index_width
        self.height = height
        self.width = width
        self.show_index = show_index

        self._df = df.copy().astype(str)
        self._cells = {}
        self._build()

    def _make_header_cell(self, value, width):
        return widgets.HTML(
            value=f"<b>{value}</b>",
            layout=widgets.Layout(
                width=width,
                min_width=width,
                max_width=width,
                padding="2px 4px",
                border="1px solid #ddd",
            ),
        )

    def _make_index_cell(self, value, width):
        return widgets.HTML(
            value=str(value),
            layout=widgets.Layout(
                width=width,
                min_width=width,
                max_width=width,
                padding="2px 4px",
                border="1px solid #ddd",
            ),
        )

    def _make_text(self, value):
        return widgets.Text(
            value=str(value),
            layout=widgets.Layout(
                width=self.cell_width,
                min_width=self.cell_width,
                max_width=self.cell_width,
            ),
        )

    def _build(self):
        self._cells = {}
        all_rows = []

        # header row INSIDE the scroll area
        header_widgets = []
        if self.show_index:
            header_widgets.append(self._make_header_cell("", self.index_width))
        for col in self._df.columns:
            header_widgets.append(self._make_header_cell(col, self.cell_width))

        header_row = widgets.HBox(
            header_widgets,
            layout=widgets.Layout(width="max-content")
        )
        all_rows.append(header_row)

        # data rows
        for r_idx, idx in enumerate(self._df.index):
            row_widgets = []
            if self.show_index:
                row_widgets.append(self._make_index_cell(idx, self.index_width))

            for c_idx, col in enumerate(self._df.columns):
                txt = self._make_text(self._df.iloc[r_idx, c_idx])
                self._cells[(r_idx, c_idx)] = txt
                row_widgets.append(txt)

            row = widgets.HBox(
                row_widgets,
                layout=widgets.Layout(width="max-content")
            )
            all_rows.append(row)

        content = widgets.VBox(
            all_rows,
            layout=widgets.Layout(width="max-content")
        )

        scroll_area = widgets.Box(
            [content],
            layout=widgets.Layout(
                overflow="auto",
                width=self.width,
                height=self.height,
                border="1px solid #bbb",
            ),
        )

        self.children = [scroll_area]

    def get_df(self) -> pd.DataFrame:
        data = []
        for r_idx in range(len(self._df.index)):
            row = []
            for c_idx in range(len(self._df.columns)):
                row.append(self._cells[(r_idx, c_idx)].value)
            data.append(row)
        return pd.DataFrame(data, index=self._df.index, columns=self._df.columns)

    @property
    def value(self):
        return self.get_df()

In [2]:
df = pd.DataFrame(
    [["a", "b", "c", 55], ["d", 22, "f",666 ]],
    index=["col1", "col2"],
    columns=[  2021, 2022,2023,2024]
)

grid = StringGrid(df, width="800px", height="150px")
display(grid)

edited = grid.value

StringGrid(children=(Box(children=(VBox(children=(HBox(children=(HTML(value='<b></b>', layout=Layout(border_bo…

In [3]:
                                                                                    grid.value

,2021,2022,2023,2024
col1,a,b,c,55
col2,d,22,f,666


In [4]:
from dataclasses import dataclass, field
from typing import Optional
import html

import ipywidgets as widgets
import pandas as pd
from IPython.display import display


@dataclass
class StringGrid:
    df: pd.DataFrame
    cell_width: str = "140px"
    index_width: str = "120px"
    height: str = "300px"
    width: str = "800px"
    show_index: bool = True
    column_tooltips: Optional[dict] = None

    _df: pd.DataFrame = field(init=False, repr=False)
    _cells: dict = field(init=False, default_factory=dict, repr=False)
    widget: widgets.Widget = field(init=False, repr=False)

    def __post_init__(self):
        self.column_tooltips = self.column_tooltips or {}
        self._df = self.df.copy().astype(str)
        self._build()

    def _make_header_cell(self, value, width, tooltip=""):
        text = html.escape(str(value))
        tip = html.escape(str(tooltip or ""))

        # Use HTML title attribute for reliable browser-native tooltip
        return widgets.HTML(
            value=(
                f'<div title="Ib Hansen {tip}" '
                f'style="font-weight:bold; padding:2px 4px; '
                f'border:1px solid #ddd; box-sizing:border-box; '
                f'white-space:nowrap; overflow:hidden; text-overflow:ellipsis;">'
                f'{text}'
                f'</div>'
            ),
            layout=widgets.Layout(
                width=width,
                min_width=width,
                max_width=width,
            ),
        )

    def _make_index_cell(self, value, width):
        text = html.escape(str(value))
        return widgets.HTML(
            value=(
                f'<div style="padding:2px 4px; border:1px solid #ddd; '
                f'box-sizing:border-box; white-space:nowrap; '
                f'overflow:hidden; text-overflow:ellipsis;">'
                f'{text}'
                f'</div>'
            ),
            layout=widgets.Layout(
                width=width,
                min_width=width,
                max_width=width,
            ),
        )

    def _make_text(self, value):
        return widgets.Text(
            value=str(value),
            layout=widgets.Layout(
                width=self.cell_width,
                min_width=self.cell_width,
                max_width=self.cell_width,
            ),
        )

    def _build(self):
        self._cells = {}
        rows = []

        # Header row inside the same scrollable area
        header_widgets = []
        if self.show_index:
            header_widgets.append(self._make_header_cell("", self.index_width, ""))

        for col in self._df.columns:
            tooltip = self.column_tooltips.get(col, "")
            header_widgets.append(
                self._make_header_cell(col, self.cell_width, tooltip)
            )

        header_row = widgets.HBox(
            header_widgets,
            layout=widgets.Layout(width="max-content")
        )
        rows.append(header_row)

        # Data rows
        for r_idx, idx in enumerate(self._df.index):
            row_widgets = []

            if self.show_index:
                row_widgets.append(self._make_index_cell(idx, self.index_width))

            for c_idx, col in enumerate(self._df.columns):
                cell = self._make_text(self._df.iloc[r_idx, c_idx])
                self._cells[(r_idx, c_idx)] = cell
                row_widgets.append(cell)

            row = widgets.HBox(
                row_widgets,
                layout=widgets.Layout(width="max-content")
            )
            rows.append(row)

        content = widgets.VBox(
            rows,
            layout=widgets.Layout(width="max-content")
        )

        self.widget = widgets.Box(
            [content],
            layout=widgets.Layout(
                overflow="auto",
                width=self.width,
                height=self.height,
                border="1px solid #bbb",
            ),
        )

    def get_df(self) -> pd.DataFrame:
        data = []
        for r_idx in range(len(self._df.index)):
            row = []
            for c_idx in range(len(self._df.columns)):
                row.append(self._cells[(r_idx, c_idx)].value)
            data.append(row)

        return pd.DataFrame(
            data,
            index=self._df.index.copy(),
            columns=self._df.columns.copy(),
        )

    @property
    def value(self) -> pd.DataFrame:
        return self.get_df()

    def set_df(self, df: pd.DataFrame):
        self.df = df
        self._df = df.copy().astype(str)
        self._build()

    def display(self):
        display(self.widget)

    def _ipython_display_(self):
        display(self.widget)

In [5]:
df = pd.DataFrame(
    [
        ["Alice", "DK", "note 1"],
        ["Bob", "SE", "note 2"],
        ["Charlie", "NO", "note 3"],
    ],
    index=["row1", "row2", "row3"],
    columns=["name", "country", "notes"],
)

tooltips = {
    "name": "Person name",
    "country": "ISO country code",
    "notes": "Free-text comments",
}

grid = StringGrid(
    df=df,
    width="500px",
    height="180px",
    cell_width="150px",
    column_tooltips=tooltips,
)

grid

Box(children=(VBox(children=(HBox(children=(HTML(value='<div title="Ib Hansen " style="font-weight:bold; paddi…